# Warp Drive Sky-Flight (Colab GPU, Drive-checkpointed)

Render high-quality videos of what the night sky looks like through
the front window of a spacecraft as it accelerates from rest into
superluminal velocities inside an Alcubierre / Natário warp bubble.

**Pipeline:**

1. Mount Google Drive — every frame is checkpointed there
2. Clone the WarpDrives repo and install in editable mode
3. Switch JAX to the GPU backend (Colab T4 / A100)
4. Download an equirectangular Milky Way panorama, or use the procedural starfield
5. For each velocity in a smooth ramp 0 → v_max, fire backward null geodesics through the warp metric and **save the PNG to Drive**
6. If the runtime times out, just rerun — the loop skips frames that already exist
7. Assemble the per-frame PNGs into MP4/GIF

Suggested runtime: **GPU (T4 or better)** — set via `Runtime → Change runtime type` before running.

## 1. Mount Google Drive (checkpoint store)

Every intermediate frame and the assembled video go into
`MyDrive/WarpDrives/skyflight/<run_name>/`.  Picking a stable
`RUN_NAME` is what makes the render *resumable*: rerun the notebook
with the same name and only the missing frames are recomputed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, json
DRIVE_ROOT = '/content/drive/MyDrive/WarpDrives'
RUN_NAME   = 'alcubierre_v0_to_4c_384px_honest_2026_04_28'   # honest-physics build (Doppler + horizon mask)
RUN_DIR    = os.path.join(DRIVE_ROOT, 'skyflight', RUN_NAME)
FRAMES_DIR = os.path.join(RUN_DIR, 'frames')
META_PATH  = os.path.join(RUN_DIR, 'meta.json')
VIDEO_DIR  = os.path.join(RUN_DIR, 'video')
for d in (RUN_DIR, FRAMES_DIR, VIDEO_DIR):
    os.makedirs(d, exist_ok=True)
print('Run directory:', RUN_DIR)

## 2. Install dependencies and clone the repo

In [ ]:
!pip install -q --upgrade "numpy>=2.0" "scipy>=1.13"
!pip install -q -U jax jaxlib
!pip install -q imageio imageio-ffmpeg pyyaml click tqdm pyvista==0.43.10

import os, sys
if not os.path.exists('/content/WarpDrives'):
    !cd /content && git clone https://github.com/mthiel74/WarpDrives.git
%cd /content/WarpDrives
!pip install -q -e .
sys.path.insert(0, os.getcwd())

In [ ]:
import jax
print('JAX:', jax.__version__)
print('Devices:', jax.devices())
print('Default backend:', jax.default_backend())

## What you're looking at

- **Camera**: pinhole, 90° FOV, attached to the bubble centre and pointing along the bubble's direction of motion (+x).
- **Velocity ramp**: cosine-eased from `v=0` to `v=V_MAX` over `N_FRAMES` frames, then mirrored back to 0 in the final video. So you start at rest, accelerate, hold briefly at top speed, decelerate.
- **Direction**: the bubble moves into +x. The Milky Way panorama is mapped to the celestial sphere with the galactic plane roughly across the equator, so initially you're looking *into* the band.
- **Doppler brightness**: honest `I_ν / ν³ = const` (Liouville's theorem). Forward pixels brighten as `f^3`, trailing pixels dim. **No fake colour shift** — we'd need spectral data to do colour honestly. **No tonemap** by default — the forward-superluminal field really does saturate in the visible band, that's not a render bug.
- **Front horizon**: rays whose backward null geodesic fails to escape the bubble are rendered black, since they have no causal contact with the celestial sphere. With the default integration budget and bubble shape this only activates for cleanly trapped rays, not for the entire `v > 1` forward cone — getting the latter requires a longer integration or a sharper bubble wall.
- **Aberration**: the Alcubierre interior observer is *naturally co-moving* with the bubble — there's no Lorentz boost between you and the warp, so the conventional SR 'headlight' beaming is absent. What you see is mild forward zoom from warp lensing on oblique rays, plus the Doppler brightness pile-up.
- **Natário caveat**: Natário's curl-based shift gives the central observer a non-timelike worldline at superluminal v. The comparison cell therefore sweeps only up to v=0.95c.
- **Sparkle / flicker**: was a sampling artefact, not physics. Mitigated by `supersample=2` (4 rays/pixel, averaged) and bilinear interpolation in `make_image_sky`.

## 3. Sky background (Drive-cached download)

The Milky Way panorama is downloaded into Drive once.  If you re-run
the notebook (or share the run with another machine) it just reuses
the cached file.

In [ ]:
import urllib.request
MW_URL  = 'https://cdn.eso.org/images/large/eso0932a.jpg'
MW_PATH = os.path.join(DRIVE_ROOT, 'assets', 'milky_way_panorama.jpg')
os.makedirs(os.path.dirname(MW_PATH), exist_ok=True)
if not os.path.exists(MW_PATH):
    print('Downloading Milky Way panorama (ESO/Brunier) into Drive…')
    urllib.request.urlretrieve(MW_URL, MW_PATH)
print('Sky image at:', MW_PATH, '|', os.path.getsize(MW_PATH) // 1024, 'KB')

In [ ]:
import numpy as np
from warpbubblesim.viz.skybackground import (
    make_image_sky, make_procedural_starfield,
)

USE_REAL_PANORAMA = os.path.exists(MW_PATH)
print('Using ESO Milky Way panorama' if USE_REAL_PANORAMA
      else 'Using procedural starfield')

## 4. Render configuration & velocity schedule

The schedule is written to `meta.json` so a resumed run uses the
**identical** velocities — important when checkpoints already exist
on Drive.

In [ ]:
from warpbubblesim.viz.skyrender_jax import (
    JaxRenderConfig, render_frame_jax,
)

RESOLUTION = 384
FOV_DEG    = 90.0
N_FRAMES   = 60
V_MAX      = 4.0
METRIC     = 'alcubierre'      # 'alcubierre' or 'natario'
BASE_PARAMS = dict(R=1.0, sigma=8.0, shape='tanh') if METRIC == 'alcubierre' \
             else dict(R=1.0, sigma=8.0)
JAX_CFG = JaxRenderConfig(
    width=RESOLUTION, height=RESOLUTION, fov_deg=FOV_DEG,
    n_steps=240, dlam=0.15, chunk_size=8192,
    supersample=2,                # 4 rays/pixel anti-aliasing
    enable_doppler=True,          # honest f³ brightness (Liouville's I_ν/ν³)
    doppler_intensity_power=3.0,  # 3 = monochromatic, 4 = bolometric thermal
    doppler_tonemap=False,        # honest physics: no tonemap
    enable_horizon_mask=True,     # blank trapped pixels at v>1
)

def make_velocities(n_frames, v_max):
    tt = np.linspace(0.0, 1.0, n_frames)
    ease = 0.5 - 0.5 * np.cos(np.pi * tt)
    return (v_max * ease).tolist()

if os.path.exists(META_PATH):
    meta = json.load(open(META_PATH))
    print('Loaded existing schedule from', META_PATH)
    velocities = meta['velocities']
    # Sanity-check that current settings match — if you changed
    # RESOLUTION/V_MAX/etc., either keep them OR pick a new RUN_NAME.
    if (meta['n_frames'] != N_FRAMES or meta['v_max'] != V_MAX
        or meta['metric'] != METRIC):
        raise SystemExit(
            f"Existing run {RUN_NAME} was rendered with different settings;"
            f" pick a new RUN_NAME (currently meta={meta})."
        )
else:
    velocities = make_velocities(N_FRAMES, V_MAX)
    meta = dict(
        run_name=RUN_NAME, metric=METRIC, base_params=BASE_PARAMS,
        resolution=RESOLUTION, fov_deg=FOV_DEG, n_frames=N_FRAMES,
        v_max=V_MAX, velocities=velocities,
        jax_cfg=dict(n_steps=JAX_CFG.n_steps, dlam=JAX_CFG.dlam,
                     chunk_size=JAX_CFG.chunk_size),
    )
    json.dump(meta, open(META_PATH, 'w'), indent=2)
    print('Wrote schedule to', META_PATH)

print(f'{len(velocities)} frames, v ∈ [{velocities[0]:.3f}, {max(velocities):.3f}]')

## 5. Render loop with per-frame Drive checkpointing

Each frame is saved as `frames/frame_<idx>_v<vel>.png` *immediately*
after it's computed.  Reruns of this cell skip frames whose PNG
already exists, so a 12-hour timeout in the middle of a sweep just
costs you the partially-rendered frame, not the run.

In [ ]:
import time, glob
import imageio.v2 as iio

def frame_path(idx):
    return os.path.join(FRAMES_DIR, f'frame_{idx:04d}.png')

def build_sky():
    if USE_REAL_PANORAMA:
        return make_image_sky(MW_PATH, rotation_deg=0.0, gain=1.6)
    return make_procedural_starfield(
        n_stars=18000, seed=2026,
        star_radius_px=0.55, fov_scale=np.deg2rad(FOV_DEG) / RESOLUTION,
    )

sky = build_sky()

n_pre  = sum(1 for i in range(N_FRAMES) if os.path.exists(frame_path(i)))
print(f'{n_pre}/{N_FRAMES} frames already on Drive — resuming from frame {n_pre}.')

rendered_this_session = 0
t_session = time.time()
for i, v in enumerate(velocities):
    fp = frame_path(i)
    if os.path.exists(fp):
        continue
    t0 = time.time()
    params = dict(BASE_PARAMS, v0=float(v))
    img = render_frame_jax(METRIC, params, sky, JAX_CFG)
    rgb = np.clip(img * 255, 0, 255).astype(np.uint8)
    # Atomic write: stage to .tmp then rename, so a crash mid-write
    # leaves no partial PNG that the resume logic would mistakenly
    # treat as a finished frame.
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb)
    os.replace(tmp, fp)
    rendered_this_session += 1
    elapsed = time.time() - t0
    sess_t  = time.time() - t_session
    remaining = N_FRAMES - i - 1
    rate = rendered_this_session / max(sess_t, 1e-6)
    eta = remaining / max(rate, 1e-6)
    print(f'  frame {i+1}/{N_FRAMES} v={v:.3f}  {elapsed:.1f}s  '
          f'(rate {rate:.2f}/s, ETA {eta/60:.1f} min)')

n_post = sum(1 for i in range(N_FRAMES) if os.path.exists(frame_path(i)))
print(f'\nDone: {n_post}/{N_FRAMES} frames on Drive.')

## 6. Assemble MP4 and GIF from disk

The video is read **from Drive**, not from in-memory state, so this
cell works in a fresh runtime as long as the frames are checkpointed.

In [ ]:
import imageio.v2 as iio

frames = []
missing = []
for i in range(N_FRAMES):
    fp = frame_path(i)
    if not os.path.exists(fp):
        missing.append(i)
        continue
    frames.append(iio.imread(fp))
if missing:
    print(f'Warning: {len(missing)} frames missing — rerun cell 5 to fill them.'
          f'  Indices: {missing[:10]}{"..." if len(missing) > 10 else ""}')

# Tail-and-back loop: pause at top speed then ramp down
extended = list(frames) + [frames[-1]] * 6 + list(reversed(frames))

MP4 = os.path.join(VIDEO_DIR, f'{RUN_NAME}.mp4')
GIF = os.path.join(VIDEO_DIR, f'{RUN_NAME}.gif')
iio.mimsave(MP4, extended, fps=24)
try:
    iio.mimsave(GIF, extended, duration=int(1000/12), loop=0)
except TypeError:
    iio.mimsave(GIF, extended, fps=12)
print('Wrote', MP4)
print('Wrote', GIF)

In [ ]:
from IPython.display import Video
Video(MP4, embed=True, width=512)

## 7. Side-by-side: Alcubierre vs Natário (also Drive-checkpointed)

Same checkpoint pattern: each composite (Alcubierre | Natário) frame
is its own PNG.  Resume by re-running.

In [ ]:
COMP_RUN_NAME = 'compare_alcubierre_natario'
COMP_RUN_DIR  = os.path.join(DRIVE_ROOT, 'skyflight', COMP_RUN_NAME)
COMP_FRAMES   = os.path.join(COMP_RUN_DIR, 'frames')
COMP_VIDEO    = os.path.join(COMP_RUN_DIR, 'video')
for d in (COMP_RUN_DIR, COMP_FRAMES, COMP_VIDEO):
    os.makedirs(d, exist_ok=True)

COMP_RES   = 256
COMP_NF    = 30
COMP_VMAX  = 0.95   # Natário central observer is not timelike at v≥1;
                    # capping the sweep keeps both metrics renderable
comp_cfg = JaxRenderConfig(width=COMP_RES, height=COMP_RES, fov_deg=FOV_DEG,
                           n_steps=240, dlam=0.15, chunk_size=8192,
                           supersample=2,
                           enable_doppler=True,
                           doppler_intensity_power=3.0,
                           doppler_tonemap=False,
                           enable_horizon_mask=True)
comp_velocities = make_velocities(COMP_NF, COMP_VMAX)

def comp_path(idx):
    return os.path.join(COMP_FRAMES, f'compare_{idx:04d}.png')

def render_comp_frame(v):
    a = render_frame_jax('alcubierre', dict(v0=float(v), R=1.0, sigma=8.0, shape='tanh'),
                         sky, comp_cfg)
    n = render_frame_jax('natario', dict(v0=float(v), R=1.0, sigma=8.0),
                         sky, comp_cfg)
    return np.concatenate([a, n], axis=1)

for i, v in enumerate(comp_velocities):
    fp = comp_path(i)
    if os.path.exists(fp):
        continue
    t0 = time.time()
    side = render_comp_frame(v)
    rgb = np.clip(side * 255, 0, 255).astype(np.uint8)
    tmp = fp + '.tmp'
    iio.imwrite(tmp, rgb)
    os.replace(tmp, fp)
    print(f'  comp {i+1}/{len(comp_velocities)} v={v:.3f} {time.time()-t0:.1f}s')

comp_frames = [iio.imread(comp_path(i))
               for i in range(len(comp_velocities))
               if os.path.exists(comp_path(i))]
COMP_MP4 = os.path.join(COMP_VIDEO, f'{COMP_RUN_NAME}.mp4')
iio.mimsave(COMP_MP4, comp_frames, fps=18)
print('Wrote', COMP_MP4)
Video(COMP_MP4, embed=True, width=600)

## Resume / restart cookbook

* **Runtime timed out mid-render?**  Re-run the notebook from the top.
  Cells 1–4 are idempotent; cell 5 reads the existing PNGs and only
  renders the missing ones.  The schedule in `meta.json` guarantees
  the same `velocities` list, so a partially-rendered run stays
  internally consistent.
* **Want a different resolution / `v_max` / metric?**  Change
  `RUN_NAME` to a fresh string in cell 1 — that gives you a new
  output directory and `meta.json`, leaving the previous renders
  intact on Drive.
* **Lost the runtime entirely?**  Frames are on Drive.  In a brand
  new runtime, just rerun: cells 1–4 reattach to Drive, cell 5
  picks up where you left off, cell 6 stitches the final video.
* **Crashed mid-write of a PNG?**  We use atomic write
  (`tmp → rename`), so an interrupted write leaves a `.tmp` file
  that the resume logic ignores.

## Notes & tips

- The renderer integrates a **fixed-step RK4**.  ``n_steps × dlam`` should
  exceed the time the slowest ray needs to leave the bubble influence;
  any extra steps are spent in flat space and don't change the
  asymptotic direction.
- ``chunk_size`` controls how many rays are vmapped together.  On a T4,
  4–8k is comfortable; on an A100 you can push it to 16–32k.
- Drive I/O is the slow part of the loop on a fast GPU — at
  ~256×256 the JAX render is ~1s, the `imwrite` to Drive ~0.2–0.5s.
  Don't use a smaller `chunk_size` to "save memory" if it makes
  rendering slower than the I/O.
- The Alcubierre interior observer is *naturally co-moving* — there's
  no SR boost between observer and bubble, so aberration in the
  conventional sense is mild.  The dramatic deformations show up at
  oblique angles where rays pass through more of the bubble wall.